In [5]:
# H3 Hexagonal Grids

import geopandas as gpd
import pandas as pd
import h3
import numpy as np
from tqdm import tqdm

print("=== H3 Hexagonal Aggregation ===")

# Load your final complete dataset
buildings = gpd.read_file("accra_buildings_solar_roi_final_complete.gpkg")
print(f"Loaded {len(buildings):,} buildings")

# Convert buildings to H3 hexagons (Resolution 9)
# H3 Resolution 9 is good balance for Accra (roughly 0.1 km² per hex)
resolution = 9

def get_h3_index(lat, lon, res=9):
    """Convert lat/lon to H3 cell index"""
    return h3.latlng_to_cell(lat, lon, res)

# Project to UTM first for accurate centroids
print("Projecting to UTM for accurate centroid calculation...")
buildings_utm = buildings.to_crs("EPSG:32630")

# Compute centroids in projected CRS
centroids_utm = buildings_utm.geometry.centroid

# Convert centroids back to WGS84 for H3
centroids = gpd.GeoSeries(centroids_utm, crs="EPSG:32630").to_crs("EPSG:4326")

print("Generating H3 indices for buildings (accurate method)...")

h3_indices = []
for pt in tqdm(centroids, total=len(centroids)):
    h3_indices.append(get_h3_index(pt.y, pt.x, resolution))

buildings["h3_index"] = h3_indices

print("H3 indexing completed.")
print(f"Sample h3_index: {buildings['h3_index'].iloc[0]}")


# Aggregate data per hexagon
# Aggregate key metrics per H3 hex
hex_agg = buildings.groupby("h3_index").agg({
    "solar_adjusted_kwh_final": "sum",
    "npv_ghs": "mean",
    "co2_savings_tonnes": "sum",
    "solar_index": "mean",
    "suitability_score": "mean",
    "building_id": "count"   # number of buildings per hex
}).reset_index()

hex_agg.rename(columns={
    "building_id": "building_count",
    "solar_adjusted_kwh_final": "total_solar_kwh",
    "npv_ghs": "avg_npv_ghs",
    "solar_index": "avg_solar_index"
}, inplace=True)

print(f"Aggregated into {len(hex_agg):,} hexagons")
print(hex_agg.head())

# Save aggregated data
hex_agg.to_csv("accra_solar_h3_aggregated.csv", index=False)
print("Saved aggregated hex data as 'accra_solar_h3_aggregated.csv'")

=== H3 Hexagonal Aggregation ===
Loaded 632,195 buildings
Projecting to UTM for accurate centroid calculation...
Generating H3 indices for buildings (accurate method)...


100%|███████████████████████████████████████████████████████████████████████████████| 632195/632195 [00:15<00:00, 39811.24it/s]


H3 indexing completed.
Sample h3_index: 8975292436bffff
Aggregated into 3,135 hexagons
          h3_index  total_solar_kwh    avg_npv_ghs  co2_savings_tonnes  \
0  89752924323ffff     2.245177e+06   64314.702440          898.070904   
1  8975292432bffff     4.038055e+06   52078.378808         1615.221851   
2  8975292432fffff     4.776306e+06   45490.189178         1910.522576   
3  89752924333ffff     4.625798e+05  123096.755456          185.031911   
4  8975292433bffff     2.896057e+06   25568.968701         1158.422824   

   avg_solar_index  suitability_score  building_count  
0        68.248874          63.945714             175  
1        63.467049          54.443714             350  
2        63.582604          55.023950             476  
3        71.846461          65.752941              17  
4        62.426368          54.711007             427  
Saved aggregated hex data as 'accra_solar_h3_aggregated.csv'


In [6]:
print("=== Converting H3 indices to polygon geometries ===")

def h3_to_polygon(h3_index):
    """Convert H3 index to Shapely Polygon"""
    boundary = h3.cell_to_boundary(h3_index)  # returns list of (lat, lon) tuples
    # Convert to (lon, lat) for Shapely
    coords = [(lon, lat) for lat, lon in boundary]
    return geom.Polygon(coords)

import shapely.geometry as geom

# Apply conversion
print("Generating hexagon geometries...")
hex_agg["geometry"] = hex_agg["h3_index"].apply(h3_to_polygon)

# Create GeoDataFrame
hex_gdf = gpd.GeoDataFrame(hex_agg, geometry="geometry", crs="EPSG:4326")

print(f"Created GeoDataFrame with {len(hex_gdf):,} hexagons")
print(hex_gdf.head())

# Save as GeoPackage for mapping
hex_gdf.to_file("accra_solar_h3_grid.gpkg", driver="GPKG")
print("Saved hexagon grid as 'accra_solar_h3_grid.gpkg'")

=== Converting H3 indices to polygon geometries ===
Generating hexagon geometries...
Created GeoDataFrame with 3,135 hexagons
          h3_index  total_solar_kwh    avg_npv_ghs  co2_savings_tonnes  \
0  89752924323ffff     2.245177e+06   64314.702440          898.070904   
1  8975292432bffff     4.038055e+06   52078.378808         1615.221851   
2  8975292432fffff     4.776306e+06   45490.189178         1910.522576   
3  89752924333ffff     4.625798e+05  123096.755456          185.031911   
4  8975292433bffff     2.896057e+06   25568.968701         1158.422824   

   avg_solar_index  suitability_score  building_count  \
0        68.248874          63.945714             175   
1        63.467049          54.443714             350   
2        63.582604          55.023950             476   
3        71.846461          65.752941              17   
4        62.426368          54.711007             427   

                                            geometry  
0  POLYGON ((-0.00142 5.64796, 

In [7]:
import geopandas as gpd
hex_gdf = gpd.read_file("accra_solar_h3_grid.gpkg")
hex_gdf["solar_per_building"] = hex_gdf["total_solar_kwh"] / hex_gdf["building_count"]
hex_gdf["log_solar"] = np.log1p(hex_gdf["total_solar_kwh"])
hex_gdf.to_file("accra_solar_h3_grid_ready.gpkg", driver="GPKG")
print("Pre-computed ready file saved!")

Pre-computed ready file saved!
